# Per-Visit Astrometric Residual Field vs PSF Moments (v1)

**Author:** Aaron Roodman
**Date Created:** 2026-08-13
**Last Modified:** 2026-08-13
**Status:** In Progress
**Keywords:** astrometry, atmospheric turbulence, wind, PSF ellipticity, WCS residuals, optical misalignment

## Description

Study the per-visit astrometric residuals of individual LSSTCam images as a probe of the
atmospheric contribution, and compare their spatial pattern across the focal plane with the
PSF ellipticity / higher-moment pattern for the same visit.

The astrometric residual measured here is the single-frame WCS solution (built on the fixed
`astrometry_camera` distortion model + a per-visit affine) differenced against the Gaia
astrometric reference catalog, **before** the Gaussian-Process turbulence correction
(`fit_turbulence`). That residual field should carry a significant, spatially-coherent
(wind-streaked) turbulence component.

Key functionality:
1. Locate and load `preliminary_visit_image`, `single_visit_star_ref_match_astrom`, and
   `single_visit_star` for one visit.
2. Build the astrometric residual vector field (measured − Gaia, mas) mapped onto the focal plane.
3. Build the PSF ellipticity field from the star second moments (`ixx/iyy/ixy`) for the same visit.
4. Compare the two fields side-by-side to begin disentangling turbulence (stochastic, wind-coherent)
   from optical misalignment (static, low-order in focal-plane coordinates).

**Output:** side-by-side focal-plane maps (astrometric residual quiver vs PSF ellipticity whisker);
per-visit residual arrays saved to `output/`.

**Based on:** DM astrometric pipeline (`GbdesAstrometricFitTask`, `BuildCameraFromAstrometryTask`,
`fit_turbulence`), analysis_tools `refCatMatchPlots`, and the moment machinery in
`rubin-work/optatmo` (`extract_psf_moments.py`, `moments_hsm.py`). C. Saunders (DM) astrometric work.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-08-13 | Aaron Roodman | Initial version — per-visit residual field + PSF ellipticity comparison |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Analysis — Residual & Ellipticity Fields](#analysis)
6. [Results & Plots](#results)
7. [Fixed Camera Model (astrometry_camera)](#astrocam)
8. [Next Steps — Rigorous Moments & Wind Test](#next)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================
VISIT        = 2026051300022          # LSSTCam visit (day_obs 20260513, seq 22)
BUTLER_REPO  = "/repo/main"
COLLECTION   = "LSSTCam/runs/nightlyValidation/67"  # set None to auto-discover
DET_FOR_WCS  = 94                      # example detector for a WCS / visitInfo peek

# star selection (mirrors analysis_tools / optatmo)
SNR_MIN      = 50.0                    # psfFlux / psfFluxErr
MAX_RESID    = 200.0                   # mas outlier clip on |residual|
EXTEND_MAX   = 0.5                     # extendedness < this => point source
NBIN_FP      = 24                      # focal-plane bins per axis for the maps
NMIN_BIN     = 3                       # min stars per FP bin to plot a vector

OUTPUT_DIR   = "../output"             # gitignored; symlinked to scratch on RSP


<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u

from lsst.daf.butler import Butler, CollectionType
from lsst.obs.lsst import LsstCam
import lsst.afw.cameraGeom as cg
from lsst.geom import Point2D, SpherePoint, degrees

# repo-root imports
sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting
setup_plotting()

camera = LsstCam.getCamera()           # full LSSTCam geometry (PIXELS <-> FOCAL_PLANE)
os.makedirs(OUTPUT_DIR, exist_ok=True)

<a id='functions'></a>
## Helper Functions

In [ ]:
def resolve_collection(butler, visit, dstypes):
    """Return the newest CHAINED run that holds all `dstypes` for `visit`.

    Restricts the search to candidate chained collections (fast) instead of the
    full "*" wildcard (which unions summaries over every collection = minutes).
    """
    import re
    chains = list(butler.registry.queryCollections(collectionTypes=CollectionType.CHAINED))
    cand = sorted(c for c in chains if re.search(r"LSSTCam/(runs|nightly|DRP)", c, re.I))
    runs = {}
    for dt in dstypes:
        runs[dt] = sorted({r.run for r in butler.registry.queryDatasets(
            dt, collections=cand, findFirst=False,
            where=f"instrument='LSSTCam' AND visit={visit}")})
    common = [r for r in runs[dstypes[0]] if all(r in runs[d] for d in dstypes)]
    return (common[-1] if common else None), runs


def make_wcs_getter(butler, visit):
    """Cached per-detector WCS via cheap component read (no pixels loaded)."""
    cache = {}
    def get_wcs(det):
        det = int(det)
        if det not in cache:
            cache[det] = butler.get("preliminary_visit_image.wcs",
                                    instrument="LSSTCam", visit=visit, detector=det)
        return cache[det]
    return get_wcs


def pix_to_fp(det, x, y):
    """Detector pixel (x, y) -> focal-plane position (mm)."""
    p = camera[int(det)].getTransform(cg.PIXELS, cg.FOCAL_PLANE).applyForward(Point2D(x, y))
    return p.getX(), p.getY()


def sky_to_fp(get_wcs, det, ra_deg, dec_deg):
    """Sky (deg) -> detector pixel (via per-detector WCS) -> focal-plane mm."""
    pix = get_wcs(det).skyToPixel(SpherePoint(ra_deg, dec_deg, degrees))
    return pix_to_fp(det, pix.getX(), pix.getY())


def ellipticity(ixx, iyy, ixy):
    """Second-moment ellipticity: e1=(ixx-iyy)/T, e2=2ixy/T, T=ixx+iyy."""
    T = ixx + iyy
    return (ixx - iyy) / T, 2 * ixy / T, T


def nmad(v):
    """Normalized median absolute deviation (robust sigma)."""
    v = np.asarray(v)
    return 1.4826 * np.median(np.abs(v - np.median(v)))


def robust_stats(v):
    """(n, median, mean, std, nmad-sigma) for a 1-D array."""
    v = np.asarray(v)
    return len(v), np.median(v), np.mean(v), np.std(v), nmad(v)


def fp_bin(x, y, u_, v_, nb=24, nmin=3):
    """Bin a vector field (u_, v_) on the focal plane; return bin centers + means."""
    H, xe, ye = np.histogram2d(x, y, bins=nb)
    U, _, _   = np.histogram2d(x, y, bins=[xe, ye], weights=u_)
    V, _, _   = np.histogram2d(x, y, bins=[xe, ye], weights=v_)
    Xc, Yc = np.meshgrid(0.5*(xe[:-1]+xe[1:]), 0.5*(ye[:-1]+ye[1:]), indexing="ij")
    m = H > nmin
    return Xc[m], Yc[m], (U/np.maximum(H, 1))[m], (V/np.maximum(H, 1))[m]

<a id='data'></a>
## Data Access

In [ ]:
butler = Butler(BUTLER_REPO)

# visit metadata (band via expandDataId — not a stored field on the visit record)
did  = butler.registry.expandDataId(instrument="LSSTCam", visit=VISIT)
vrec = did.records["visit"]
BAND, PHYS, DAYOBS = did["band"], did["physical_filter"], vrec.day_obs
print(f"visit {VISIT}: band={BAND} filter={PHYS} day_obs={DAYOBS}")

# collection: use the parameter, or auto-discover
DSTYPES = ["single_visit_star_ref_match_astrom", "single_visit_star", "preliminary_visit_image"]
if COLLECTION is None:
    COLLECTION, runs = resolve_collection(butler, VISIT, DSTYPES)
    print("discovered runs:", {k: v[-2:] for k, v in runs.items()})
assert COLLECTION, "no collection holds these products for this visit"
print("USING COLLECTION:", COLLECTION)

butler = Butler(BUTLER_REPO, collections=[COLLECTION])
get_wcs = make_wcs_getter(butler, VISIT)

# example WCS + visitInfo (boresight / rotator / parallactic angle for the wind frame)
pvi = butler.get("preliminary_visit_image",
                 instrument="LSSTCam", visit=VISIT, detector=DET_FOR_WCS)
vi = pvi.getInfo().getVisitInfo()
print("boresight :", vi.getBoresightRaDec())
print("rotAngle  :", vi.getBoresightRotAngle().asDegrees(), "deg")
print("parAngle  :", vi.getBoresightParAngle().asDegrees(), "deg")

<a id='analysis'></a>
## Analysis — Residual & Ellipticity Fields

In [ ]:
# ---- per-visit astrometric residuals (measured - Gaia), pre-GP ----
rm = butler.get("single_visit_star_ref_match_astrom",
                instrument="LSSTCam", visit=VISIT).to_pandas()
print("ref_match columns:", list(rm.columns))

cosd = np.cos(np.deg2rad(rm["coord_dec_target"].to_numpy()))
dX = ((rm["coord_ra_target"]  - rm["ra_ref"]).to_numpy() * cosd * u.deg).to(u.mas).value  # RA*cos(dec)
dY = ((rm["coord_dec_target"] - rm["dec_ref"]).to_numpy()        * u.deg).to(u.mas).value  # Dec

# quality selection WITHOUT the residual-magnitude clip, so we can see the tails
qual  = (rm["extendedness_target"] < EXTEND_MAX).to_numpy()
qual &= (rm["psfFlux_target"] / rm["psfFluxErr_target"]).to_numpy() > SNR_MIN
qual &= ~rm["psfFlux_flag_target"].to_numpy() & ~rm["centroid_flag_target"].to_numpy()
qual &= np.isfinite(dX) & np.isfinite(dY)

# focal-plane position for every quality star (mapped once)
fp = np.array([sky_to_fp(get_wcs, d, ra, dec) for d, ra, dec in
               zip(rm["detector_target"][qual],
                   rm["coord_ra_target"][qual], rm["coord_dec_target"][qual])])
fx_q, fy_q = fp[:, 0], fp[:, 1]
dX_q, dY_q = dX[qual], dY[qual]            # pre-clip, quality stars (for histograms)

# residual-magnitude clip -> the field used in the maps
clip = np.hypot(dX_q, dY_q) < MAX_RESID
fx, fy, dRA, dDec = fx_q[clip], fy_q[clip], dX_q[clip], dY_q[clip]
print(f"{qual.sum()} quality stars; {clip.sum()} kept after |resid|<{MAX_RESID} mas "
      f"({100*(1-clip.mean()):.1f}% clipped)")
print(f"kept residual RMS  X={dRA.std():.1f}  Y={dDec.std():.1f} mas  |  "
      f"NMAD  X={nmad(dRA):.1f}  Y={nmad(dDec):.1f} mas")

In [ ]:
# ---- residual distribution diagnostics: check tails before binning ----
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))

for a, d, lab, col in [(ax[0], dX_q, "ΔX  (RA·cosδ)", "C0"),
                       (ax[1], dY_q, "ΔY  (Dec)",     "C3")]:
    n, med, mean, std, s = robust_stats(d)
    rng = min(MAX_RESID, 5*max(s, 1))
    a.hist(d, bins=120, range=(-rng, rng), color=col, alpha=0.8)
    a.axvline(-MAX_RESID, ls="--", c="k", lw=1); a.axvline(MAX_RESID, ls="--", c="k", lw=1)
    a.axvline(med, ls=":", c="k", lw=1)
    a.set_title(f"{lab}\nmed={med:.1f}  NMAD={s:.1f}  std={std:.1f} mas  (n={n})")
    a.set_xlabel("residual (mas)"); a.set_yscale("log")

# 2-D residual scatter to expose correlated tails / systematics
ax[2].hexbin(dX_q, dY_q, gridsize=60, bins="log", cmap="viridis",
             extent=(-MAX_RESID, MAX_RESID, -MAX_RESID, MAX_RESID))
ax[2].axhline(0, c="w", lw=0.5); ax[2].axvline(0, c="w", lw=0.5)
ax[2].set_aspect("equal"); ax[2].set_xlabel("ΔX (mas)"); ax[2].set_ylabel("ΔY (mas)")
ax[2].set_title("ΔX vs ΔY (log density)")
plt.savefig(f"{OUTPUT_DIR}/astrometry_residual_hist_{VISIT}.png", dpi=120)
plt.show()

# radial-residual histogram + suggested clip diagnostics
r = np.hypot(dX_q, dY_q)
print(f"|resid| percentiles (mas): "
      f"50%={np.percentile(r,50):.1f}  90%={np.percentile(r,90):.1f}  "
      f"99%={np.percentile(r,99):.1f}  max={r.max():.1f}")

In [ ]:
# ---- PSF ellipticity field for the SAME visit (star 2nd moments) ----
# single_visit_star stores 2nd moments as ixx/iyy/ixy (same columns optatmo reads).
svs = butler.get("single_visit_star", instrument="LSSTCam", visit=VISIT).to_pandas()
print("shape/moment cols:", [c for c in svs.columns
      if any(k in c.lower() for k in ("ixx","iyy","ixy","moment"))][:20])

snr_s = svs.psfFlux / svs.psfFluxErr
ok = (svs.get("detect_isPrimary", True) & (svs.extendedness < EXTEND_MAX)
      & (snr_s > SNR_MIN) & ~svs.pixelFlags_saturated & ~svs.pixelFlags_bad
      & np.isfinite(svs.ixx) & np.isfinite(svs.ixy) & (svs.ixx + svs.iyy > 0)).to_numpy()

e1, e2, T = ellipticity(svs.ixx.to_numpy(), svs.iyy.to_numpy(), svs.ixy.to_numpy())
sfp = np.array([pix_to_fp(d, x, y) for d, x, y in
                zip(svs.detector[ok], svs.x[ok], svs.y[ok])])
e1, e2 = e1[ok], e2[ok]
print(f"{ok.sum()} PSF stars; median |e|={np.median(np.hypot(e1,e2)):.3f}, "
      f"median FWHM~{np.median(np.sqrt(T[ok]))*0.2*2.355:.2f} px-sigma-scaled")

<a id='results'></a>
## Results & Plots

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 7.5), sharex=True, sharey=True)

# left: astrometric residual quiver (measured - Gaia), colored by magnitude
X, Y, U, V = fp_bin(fx, fy, dRA, dDec, nb=NBIN_FP, nmin=NMIN_BIN)
q = ax[0].quiver(X, Y, U, V, np.hypot(U, V), cmap="viridis", angles="xy",
                 scale_units="xy")
plt.colorbar(q, ax=ax[0], fraction=0.046, label="|residual| (mas)")
ax[0].set_title(f"astrometric residual (measured − Gaia)\nvisit {VISIT}  {BAND}-band  (pre-GP)")

# right: PSF ellipticity whiskers (headless, angle = 0.5*atan2(e2,e1))
X2, Y2, E1, E2 = fp_bin(sfp[:, 0], sfp[:, 1], e1, e2, nb=NBIN_FP, nmin=NMIN_BIN)
emag, ang = np.hypot(E1, E2), 0.5*np.arctan2(E2, E1)
w = ax[1].quiver(X2, Y2, emag*np.cos(ang), emag*np.sin(ang), emag, cmap="magma",
                 headwidth=1, headlength=0, pivot="mid", angles="xy")
plt.colorbar(w, ax=ax[1], fraction=0.046, label="|e|")
ax[1].set_title(f"PSF ellipticity whisker\nvisit {VISIT}  {BAND}-band")

for a in ax:
    a.set_aspect("equal"); a.set_xlabel("focal-plane x (mm)")
ax[0].set_ylabel("focal-plane y (mm)")
plt.savefig(f"{OUTPUT_DIR}/astrometry_residual_vs_ellipticity_{VISIT}.png", dpi=120)
plt.show()

In [ ]:
# persist the per-visit residual field for downstream (structure fn / 2-pt / wind)
np.savez(f"{OUTPUT_DIR}/astrometry_residual_field_{VISIT}.npz",
         fx=fx, fy=fy, dRA=dRA, dDec=dDec,
         sfp_x=sfp[:, 0], sfp_y=sfp[:, 1], e1=e1, e2=e2,
         band=str(BAND), rot_deg=vi.getBoresightRotAngle().asDegrees(),
         par_deg=vi.getBoresightParAngle().asDegrees())
print("saved residual field npz")

<a id='astrocam'></a>
## Fixed Camera Model (astrometry_camera)

`astrometry_camera` is the static per-detector distortion model produced by
`BuildCameraFromAstrometryTask` (a subtask of `GbdesAstrometricFitTask`) from the gbdes
joint astrometric fit. It captures the **fixed astrometric features of the camera**, and is
fed back into single-frame processing (`calibrateImage`, `useButlerCamera=True`) as the
starting distortion for the per-visit WCS fit. It is a calibration dataset, one per band.

In [ ]:
cam_ref = list(butler.registry.queryDatasets(
    "astrometry_camera", collections="*", findFirst=False,
    where=f"instrument='LSSTCam' AND physical_filter='{PHYS}'"))[0]
astro_cam = butler.get(cam_ref)
print(f"astrometry_camera run: {cam_ref.run}")
print(f"{len(astro_cam)} detectors; e.g. detector {DET_FOR_WCS}:", astro_cam[DET_FOR_WCS].getName())

# compare fixed-camera distortion vs nominal obs geometry at detector center (arcsec)
det = astro_cam[DET_FOR_WCS]
bbox = det.getBBox()
cen = Point2D(bbox.getCenterX(), bbox.getCenterY())
fa_astro = det.getTransform(cg.PIXELS, cg.FIELD_ANGLE).applyForward(cen)
fa_nom   = camera[DET_FOR_WCS].getTransform(cg.PIXELS, cg.FIELD_ANGLE).applyForward(cen)
d_as = np.degrees(np.hypot(fa_astro.getX()-fa_nom.getX(),
                           fa_astro.getY()-fa_nom.getY())) * 3600
print(f"astrometry_camera vs nominal field-angle offset at center: {d_as:.3f} arcsec")

<a id='next'></a>
## Next Steps — Rigorous Moments & Wind Test

**Higher moments (3rd/4th).** The quick ellipticity above uses the catalog `ixx/iyy/ixy`
second moments. For 3rd moments (coma-like, the discriminator against optical misalignment)
reuse the optatmo estimator, which cuts 33×33 stamps off `preliminary_visit_image` and runs
the PIFF-port HSM moments:

```python
# rubin-work/optatmo/code/extract_psf_moments.py  ->  psfmoments_<visit>.parquet
#   columns: e0,e1,e2, M21,M12,M30,M03, M22,... + thx_ccs_deg, thy_ccs_deg, rot_deg
from optatmo.code.moments_hsm import measure_hsm_moments
```

Run it for this visit (same collection) and join on the focal-plane grid to compare the
3rd-moment pattern with the astrometric residual field.

**Wind test (turbulence vs optics).** Turbulence streaks are coherent along the wind
direction **on sky / in alt-az**; optical misalignment patterns are static in **focal-plane**
coordinates. Rotate the residual vectors `(dRA, dDec)` into an alt-az frame using the
`rotAngle` / `parAngle` saved in the npz, then look for a preferred elongation axis and
compute the 2-point correlation (E/B-mode split, as `fit_turbulence` does). The persistent,
low-order focal-plane pattern (averaged over many visits) is optics; single-visit-minus-mean
is atmosphere and should correlate spatially between the astrometric residual and the PSF
moment fields.

**Multi-visit.** Loop this notebook over a night's in-focus visits to separate the static
(optical) component from the visit-varying (atmospheric) component.